# Fitting a Series

In this example, we fit the `PhaseModel` to every measurement of a `PupilSeries` and extract some metrics.

In [ ]:
import matplotlib.pyplot as plt
import piprkit

## Splitting and baseline correction

First, the series is loaded and split into a `list` of `PupilMeasurement` as in previous examples.

In [ ]:
ps = piprkit.load_real_series()
pupil_measurements = ps.split(pre_stimulus_duration=10.0, post_stimulus_duration=55.0)
for measurement in pupil_measurements:
    measurement.filters.apply_baseline_correction(duration=5.0)

## Fitting each measurement

Using list comprehension the `PhaseModel` can be quickly fitted to every `PupilMeasurement`.

In [ ]:
models = [piprkit.PhaseModel(measurement, baseline_duration=5.0) for measurement in pupil_measurements]
print(f"Fitted {len(models)} measurements")

## The fits

This `list` of models is plotted using a loop, similar to plots in previous examples.

In [ ]:
fig, axes = plt.subplots(
    len(models),
    1,
    figsize=(8, 2.5 * len(models)),
    sharex=True,
    sharey=True,
)

if len(models) == 1:
    axes = [axes]  # Make it iterable for single subplot

for i, (ax, model) in enumerate(zip(axes, models)):
    model.plot(ax=ax, scatter=True, s=2, color="grey", label=f"Measurement {i + 1}")
    model.plot_sections(ax=ax)
    model.plot_phases(ax=ax)
    model.plot_light_stimulus(ax=ax)

    ax.set_title(f"Measurement {i + 1} - Data and Phase Model")
    ax.set_ylabel("Relative Pupil Size")
    ax.legend(fontsize=7, loc="lower right")
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Time (s)")
plt.tight_layout()
plt.show()

## Parameters, flags and goodness of fit

Parameters for each fit can be extracted. In this example, the goodness of fit (`RMSE`), any flags indicating potential fitting issues, and the parameters are printed.

In [ ]:
for i, model in enumerate(models):
    print(f"Measurement {i + 1}: RMSE {model.get_goodness_of_fit('RMSE'):.3f}")
    flags = model.get_flags()
    for phase, params in model.get_all_params().items():
        values = ", ".join(f"{name} = {value:.3f}" for name, value in params.items())
        reason = flags[phase]["reason"]
        print(f"  {phase}: {values}" + (f"  ({reason})" if reason else ""))

## 6s-PIPR

Using `to_measurement` each model is changed to a `PupilMeasurement` object, so that metrics can be extracted. This is compared to the 6s-PIPR before the fit was made.

In [ ]:
for i, (measurement, model) in enumerate(zip(pupil_measurements, models)):
    data_pipr = measurement.metrics.pipr_6s()
    fit_pipr = model.to_measurement().metrics.pipr_6s()
    print(f"Measurement {i + 1}: {data_pipr.get_name()} data {data_pipr:.3f}, fit {fit_pipr:.3f}")

## All trials as one long table

Using `piprkit.metric_rows` the metrics are prepared as long-data format rows. Using `piprkit.save_metric_rows` the data is stored in a file.

In [ ]:
rows = []
for trial, (measurement, model) in enumerate(zip(pupil_measurements, models), start=1):
    results = [
        measurement.metrics.pipr_6s(),
        measurement.metrics.auc_early(baseline_duration=5.0),
        model.metrics.plateau_pipr(),
    ]
    rows += piprkit.metric_rows(results, participant="example", trial=trial)

for row in rows[:3]:
    print(row)